# Модуль 2. Диффузионные модели. Часть 2.
##Семинар №3.Изучение особенностей диффузионных моделей с библиотекой accelerate.

### Цель занятия
Рассмотреть библиотеку accelerate для обучения, дообучения диффузионных моделей и поработать с ней.



---


### Перенос вашего кода в Accelerate

В этом руководстве подробно описано, как легко преобразовать существующий код PyTorch для использования Accelerate. Вы увидите, что, просто изменив несколько строк кода, Accelerate может творить чудеса и помочь вам с легкостью запускать свой код в распределенных системах.

### Базовый цикл обучения

Для начала напишем очень простой цикл обучения PyTorch.

> Мы предполагаем, что `training_dataloader`, `model`, `optimizer`, `scheduler` и `loss_function` были определены заранее.

```
device = "cuda"
model.to(device)

for batch in training_dataloader:
    optimizer.zero_grad()
    inputs, targets = batch
    inputs = inputs.to(device)
    targets = targets.to(device)
    outputs = model(inputs)
    loss = loss_function(outputs, targets)
    loss.backward()
    optimizer.step()
    scheduler.step()
```

### Добавляем Accelerate

Чтобы начать использовать Accelerate, сначала импортируйте и создайте экземпляр Accelerator:

```
from accelerate import Accelerator

accelerator = Accelerator()
```

Accelerator — это основная сила, позволяющая использовать все возможные варианты распределенного обучения!

#### Настройка правильного устройства

Класс Accelerator знает, на какое устройство можно в любой момент переместить любой объект PyTorch, поэтому вам следует изменить определение устройства, чтобы оно исходило из Accelerator:

```
- device = 'cuda'
+ device = accelerator.device
  model.to(device)
```

#### Подготовка объектов

Далее нужно передать все важные объекты, связанные с обучением, в `prepare()`. Accelerate позаботится о том, чтобы все было настроено в текущей среде, чтобы вы могли начать обучение:

```python
model, optimizer, training_dataloader, scheduler = accelerator.prepare(
    model, optimizer, training_dataloader, scheduler
)
```

Эти объекты возвращаются в том же порядке, в котором они были отправлены. По умолчанию при использовании `device_placement=True` все объекты, которые можно отправить на нужное устройство, будут возвращены. Если вам нужно работать с данными, которые не передаются в [~Accelerator.prepare], но должны находиться на активном устройстве, вам следует передать устройство, созданное вами ранее.

> Accelerate подготавливает только объекты, которые наследуются от соответствующих классов PyTorch (например, torch.optim.Optimizer).

#### Изменение цикла обучения

Наконец, в цикле обучения необходимо изменить три строки кода. Классы `DataLoader` Accelerate по умолчанию автоматически обрабатывают размещение устройства, а для выполнения обратного прохода следует использовать метод `backward()`:

```python
outputs = model(inputs)
loss = loss_function(outputs, targets)
accelerator.backward(loss)

# Вместо:
# inputs = inputs.to(device)
# targets = targets.to(device)
# outputs = model(inputs)
# loss = loss_function(outputs, targets)
# loss.backward()
```

Теперь ваш тренировочный цикл готов к использованию Accelerate!

#### Конечный код

Ниже приведена окончательная версия преобразованного кода:

In [ ]:
%%capture
!pip install -q -U einops datasets matplotlib tqdm numpy torch torchvision transformers accelerate

In [ ]:
from accelerate import Accelerator

accelerator = Accelerator()

model, optimizer, training_dataloader, scheduler = accelerator.prepare(
    model, optimizer, training_dataloader, scheduler
)

for batch in training_dataloader:
    optimizer.zero_grad()
    inputs, targets = batch
    outputs = model(inputs)                 # <--
    loss = loss_function(outputs, targets)  # <--
    accelerator.backward(loss)              # <--
    optimizer.step()
    scheduler.step()

Но как запустить этот код и заставить его использовать специальное оборудование, доступное для него?

Во-первых, следует переписать приведенный выше код в функцию и сделать ее вызываемой как скрипт. Например:

In [ ]:
from accelerate import Accelerator

def main():
    accelerator = Accelerator()

    model, optimizer, training_dataloader, scheduler = accelerator.prepare(
        model, optimizer, training_dataloader, scheduler
    )

    for batch in training_dataloader:
        optimizer.zero_grad()
        inputs, targets = batch
        outputs = model(inputs)
        loss = loss_function(outputs, targets)
        accelerator.backward(loss)
        optimizer.step()
        scheduler.step()

if __name__ == "__main__":
    main()

UnboundLocalError: local variable 'model' referenced before assignment

#### Использование запуска Accelerate

В Accelerate есть специальная команда CLI, которая поможет вам запустить код в вашей системе посредством accelerate launch. Эта команда охватывает все различные команды, необходимые для запуска вашего скрипта на различных платформах, и вам не нужно запоминать, что представляет собой каждая из них.

Можно быстро запустить свой скрипт, используя:

In [ ]:
!accelerate launch {script_name.py} --arg1 --arg2 ...

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
/usr/bin/python3: can't open file '/content/{script_name.py}': [Errno 2] No such file or directory
Traceback (most recent call last):
  File "/usr/local/bin/accelerate", line 8, in <module>
    sys.exit(main())
  File "/usr/local/lib/python3.10/dist-packages/accelerate/commands/accelerate_cli.py", line 48, in main
    args.func(args)
  File "/usr/local/lib/python3.10/dist-packages/accelerate/commands/launch.py", line 1168, in launch_command
    simple_launcher(args)
  File "/usr/local/lib/python3.10/dist-packages/accelerate/commands/launch.py", line 763, in simple_launcher
    raise subprocess.CalledProcessErr

Просто поместите `accelerate launch` в начало вашей команды, а затем передайте дополнительные аргументы и параметры в ваш скрипт, как обычно.

Здесь также можно изменить все ожидаемые переменные среды. Например, вот как использовать `accelerate launch` с одним GPU:

In [ ]:
!export CUDA_VISIBLE_DEVICES="0"
!accelerate launch {script_name.py} --arg1 --arg2 ...

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
/usr/bin/python3: can't open file '/content/{script_name.py}': [Errno 2] No such file or directory
Traceback (most recent call last):
  File "/usr/local/bin/accelerate", line 8, in <module>
    sys.exit(main())
  File "/usr/local/lib/python3.10/dist-packages/accelerate/commands/accelerate_cli.py", line 48, in main
    args.func(args)
  File "/usr/local/lib/python3.10/dist-packages/accelerate/commands/launch.py", line 1168, in launch_command
    simple_launcher(args)
  File "/usr/local/lib/python3.10/dist-packages/accelerate/commands/launch.py", line 763, in simple_launcher
    raise subprocess.CalledProcessErr

Вы также можете использовать `accelerate launch` без предварительного конфига accelerate, но вам может потребоваться вручную передать правильные параметры конфигурации. В этом случае Accelerate примет за вас некоторые решения по гиперпараметрам, например, если GPU доступны, он будет использовать их все по умолчанию без смешанной точности. Вот как можно использовать все GPU и тренироваться с отключенной смешанной точностью:

In [ ]:
accelerate launch --multi_gpu {script_name.py} {--arg1} {--arg2} ...

Или указав количество используемых GPU:

In [ ]:
accelerate launch --num_processes=2 {script_name.py} {--arg1} {--arg2} ...

Чтобы получить более конкретную информацию, вам следует передать необходимые параметры самостоятельно. Например, вот как можно запустить тот же сценарий на двух графических процессорах со смешанной точностью, избегая при этом всех предупреждений:

In [ ]:
accelerate launch --multi_gpu --mixed_precision=fp16 --num_processes=2 {script_name.py} {--arg1} {--arg2} ...

Чтобы получить полный список параметров, которые вы можете передать, запустите:

In [ ]:
!accelerate launch -h

Для визуализации этой разницы более ранний запуск `accelerate` на нескольких GPU с torchrun выглядел бы примерно так:

In [ ]:
MIXED_PRECISION="fp16" torchrun --nproc_per_node=2 --num_machines=1 {script_name.py} {--arg1} {--arg2} ...

Вы также можете запустить свой скрипт, используя CLI запуск в качестве самого модуля Python, что позволяет передавать другие варианты поведения запуска, специфичные для Python. Для этого используйте `accelerate.commands.launch` вместо «accelerate launch»:

In [ ]:
python -m accelerate.commands.launch --num_processes=2 {script_name.py} {--arg1} {--arg2}

Если вы хотите выполнить сценарий с любыми другими флагами python, вы можете передать их так же, как и -m, например, как в приведенном ниже примере, включающем небуферизованный стандартный вывод и стандартный поток ошибок:

In [ ]:
python -u -m accelerate.commands.launch --num_processes=2 {script_name.py} {--arg1} {--arg2}

#### Почему вы всегда должны использовать accelerate config

Почему это настолько полезно, что вам всегда нужно запускать `accelerate config`?

Помните тот предыдущий вызов `accelerate launch`, а также запуск `torchrun`? После настройки, чтобы запустить этот скрипт с необходимыми частями, вам просто нужно сразу использовать `accelerate launch`, не передавая ничего больше:

In [ ]:
accelerate launch {script_name.py} {--arg1} {--arg2} ...

#### Пользовательские конфигурации

Как кратко упоминалось ранее, ускоренный запуск следует в основном использовать путем объединения настроек набора, созданных с помощью команды ускорения конфигурации. Эти конфигурации сохраняются в файле `default_config.yaml` в папке кэша для `Accelerate`. Эта папка кэша находится по адресу (в порядке убывания приоритета):

К содержимому вашей переменной среды `HF_HOME` добавлен суффикс ускорения.
Если он не существует, к содержимому вашей переменной среды `XDG_CACHE_HOME` добавляется суффикс `huggingface/accelerate`.
Если и его не существует, папка `~/.cache/huggingface/accelerate`.
Чтобы иметь несколько конфигураций, флаг `--config_file` можно передать команде ускоренного запуска вместе с расположением пользовательского файла `yaml`.

Пример `yaml` может выглядеть примерно так для двух GPU на одной машине с использованием `fp16` для смешанной точности:

In [ ]:
compute_environment: LOCAL_MACHINE
deepspeed_config: {}
distributed_type: MULTI_GPU
fsdp_config: {}
machine_rank: 0
main_process_ip: null
main_process_port: null
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 2
use_cpu: false

Запуск сценария из местоположения этого пользовательского файла `yaml` выглядит следующим образом:

```shell
accelerate launch --config_file path/to/config/my_config_file.yaml {script_name.py} {--arg1} {--arg2} ...
```

## Запуск обучения на нескольких узлах

### Настройка среды

Прежде чем можно будет выполнить какое-либо обучение, в системе должен существовать файл accelerate config. Обычно это можно сделать, выполнив в терминале следующую команду:
```
accelerate config
```
Однако, если общие настройки по умолчанию подходят и вы не используете TPU, у ускорения есть утилита для быстрой записи конфигурации вашего GPU в файл конфигурации через `write_basic_config`.

Следующая ячейка перезапустит Jupyter после записи конфигурации, поскольку для этого был вызван код CUDA. CUDA не может быть инициализирован более одного раза (один раз для ноутбуков с одним GPU, используемых по умолчанию, а затем еще раз при вызове `notebook_launcher`). Можно выполнять отладку в блокноте и вызывать CUDA, но помните, что для окончательного обучения необходимо выполнить полную очистку и перезапуск, как показано ниже:



In [ ]:
import os
from accelerate.utils import write_basic_config
write_basic_config() # Write a config file
# os._exit(0) # Restart the notebook

Configuration already exists at /root/.cache/huggingface/accelerate/default_config.yaml, will not override. Run `accelerate config` manually or pass a different `save_location`.


False

In [ ]:
!cat /root/.cache/huggingface/accelerate/default_config.yaml

{
  "compute_environment": "LOCAL_MACHINE",
  "debug": false,
  "distributed_type": "NO",
  "downcast_bf16": false,
  "enable_cpu_affinity": false,
  "machine_rank": 0,
  "main_training_function": "main",
  "mixed_precision": "no",
  "num_machines": 1,
  "num_processes": 1,
  "rdzv_backend": "static",
  "same_network": false,
  "tpu_use_cluster": false,
  "tpu_use_sudo": false,
  "use_cpu": false
}


### Подготовка датасета и модели

Далее нужно подготовить набор данных. Как упоминалось ранее, при подготовке `DataLoaders` и модели следует проявлять особую осторожность, чтобы убедиться, что ни на один GPU данные внутри не переносятся.

Если вы это сделаете, рекомендуется поместить этот конкретный код в функцию и вызывать ее из интерфейса запуска ноутбука, который будет показан позже.

Убедитесь, что датасет загружен в соответствии с инструкциями [здесь](https://github.com/huggingface/accelerate/tree/main/examples#simple-vision-example).

In [ ]:
#импорт 1 мин 15 секунд
%%capture
!pip install accelerate matplotlib timm tqdm  google-api-python-client>=1.12.5
!pip install evaluate datasets==2.3.2 dill==0.3.5.1 evaluate==0.4.1 huggingface-hub==0.19.4 torch==2.1.2 multiprocess==0.70.13
!pip install torchvision responses==0.18.0 tokenizers transformers==4.36.2 xxhash==3.4.1
! pip install cloud-tpu-client==0.10 diffusers
! pip install https://storage.googleapis.com/tpu-pytorch/wheels/colab/torch_xla-2.0-cp310-cp310-linux_x86_64.whl
#! pip install git+https://github.com/huggingface/accelerate
!wget https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz
!tar -xzf images.tar.gz

Сначала мы создадим функцию для извлечения имени класса на основе файла:

In [ ]:
import os
data_dir = "/content/images"
fnames = os.listdir(data_dir)
fname = fnames[0]
print(fname)

english_setter_165.jpg


In [ ]:
import os, re, torch, PIL
import numpy as np
from tqdm.auto import tqdm
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import Compose, RandomResizedCrop, Resize, ToTensor

from accelerate import Accelerator
from accelerate.utils import set_seed
from timm import create_model

В данном случае метка — `keeshond`:

In [ ]:
import re
def extract_label(fname):
    stem = fname.split(os.path.sep)[-1]
    return re.search(r"^(.*)_\d+\.jpg$", stem).groups()[0]

In [ ]:
extract_label(fname)

'english_setter'

Дальше создаем класс `Dataset`

In [ ]:
class PetsDataset(Dataset):
    def __init__(self, file_names, image_transform=None, label_to_id=None):
        self.file_names = file_names
        self.image_transform = image_transform
        self.label_to_id = label_to_id

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        fname = self.file_names[idx]
        raw_image = PIL.Image.open(fname)
        image = raw_image.convert("RGB")
        if self.image_transform is not None:
            image = self.image_transform(image)
        label = extract_label(fname)
        if self.label_to_id is not None:
            label = self.label_to_id[label]
        return {"image": image, "label": label}

И создадим наш набор данных

In [ ]:
# Захватываем все имена файлов изображений
fnames = [ os.path.join(data_dir, fname)
    for fname in fnames
    if fname.endswith(".jpg")
]

# Создаем метки
all_labels = [
    extract_label(fname)
    for fname in fnames
]
id_to_label = list(set(all_labels))
id_to_label.sort()
label_to_id = {lbl: i for i, lbl in enumerate(id_to_label)}



> Примечание: это будет храниться внутри функции, поскольку мы будем устанавливать `seed` во время обучения.



In [ ]:
def get_dataloaders(batch_size:int=64):
    "Создает множество dataloaders с batch_size"
    random_perm = np.random.permutation(len(fnames))
    cut = int(0.8 * len(fnames))
    train_split = random_perm[:cut]
    eval_split = random_perm[:cut]

    # Для обучения мы используемы обычный RandomResizedCrop
    train_tfm = Compose([
        RandomResizedCrop((224, 224), scale=(0.5, 1.0)),
        ToTensor()
    ])
    train_dataset = PetsDataset(
        [fnames[i] for i in train_split],
        image_transform=train_tfm,
        label_to_id=label_to_id
    )

    # Для eval мы используем детерминированный Resize
    eval_tfm = Compose([
        Resize((224, 224)),
        ToTensor()
    ])
    eval_dataset = PetsDataset(
        [fnames[i] for i in eval_split],
        image_transform=eval_tfm,
        label_to_id=label_to_id
    )

    # Создание экземпляров dataloader
    train_dataloader = DataLoader(
        train_dataset,
        shuffle=True,
        batch_size=batch_size,
        num_workers=4
    )
    eval_dataloader = DataLoader(
        eval_dataset,
        shuffle=False,
        batch_size=batch_size*2,
        num_workers=4
    )
    return train_dataloader, eval_dataloader

### Написание функции обучения

Теперь мы можем построить наш цикл обучения. `notebook_launcher` работает, передавая функцию для вызова, которая будет выполняться в распределенной системе.

Вот базовый цикл обучения для нашей задачи классификации животных:

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR

In [ ]:
def training_loop(mixed_precision='no', seed:int=42, batch_size:int=64):
    set_seed(seed)
    # Создаем accelerator
    global accelerator
    accelerator.print("*"*40," accelerator.distributed_type",accelerator.distributed_type)
    # Создаем dataloaders
    train_dataloader, eval_dataloader = get_dataloaders(batch_size)

    # Создаем экземпляр модели (здесь мы строим модель так, чтобы начальное число также контролировало новые инициализации веса)
    model = create_model("resnet50d", pretrained=True, num_classes=len(label_to_id))

    # Замораживаем базовую модель
    for param in model.parameters():
        param.requires_grad=False
    for param in model.get_classifier().parameters():
        param.requires_grad=True

    # Нормализуем батчи изображений чтобы было более быстро
    mean = torch.tensor(model.default_cfg["mean"])[None, :, None, None]
    std = torch.tensor(model.default_cfg["std"])[None, :, None, None]

    # Чтобы сделать эту константу доступной на активном устройстве, мы устанавливаем ее на accelerator устройство.
    mean = mean.to(accelerator.device)
    std = std.to(accelerator.device)

    # Создаем экземпляр оптимизатора
    optimizer = torch.optim.Adam(params=model.parameters(), lr = 3e-2/25)

    # Создаем learning rate scheduler
    lr_scheduler = OneCycleLR(
        optimizer=optimizer,
        max_lr=3e-2,
        epochs=1,
        steps_per_epoch=len(train_dataloader)
    )

    # Подготовка
    # Нет определенного порядка, который нужно запомнить,
    # нам просто нужно распаковать объекты в том же порядке,
    # в котором мы указали их в методе подготовки..
    model, optimizer, train_dataloader, eval_dataloader, lr_scheduler = accelerator.prepare(
        model, optimizer, train_dataloader, eval_dataloader, lr_scheduler
    )

    # Теперь тренируем модель
    for epoch in tqdm(range(1)):
        model.train()
        for step, batch in enumerate(train_dataloader):
            # Мы могли бы избежать этой строки, поскольку установили accelerator с помощью device_placement=True.
            batch = {k: v.to(accelerator.device) for k, v in batch.items()}
            inputs = (batch["image"] - mean) / std
            outputs = model(inputs)
            loss = torch.nn.functional.cross_entropy(outputs, batch["label"])
            accelerator.backward(loss)
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()

        model.eval()
        accurate = 0
        num_elems = 0
        for _, batch in enumerate(eval_dataloader):
            # Мы могли бы избежать этой строки, поскольку установили accelerator с помощью device_placement=True.
            batch = {k: v.to(accelerator.device) for k, v in batch.items()}
            inputs = (batch["image"] - mean) / std
            with torch.no_grad():
                outputs = model(inputs)
            predictions = outputs.argmax(dim=-1)
            accurate_preds = accelerator.gather(predictions) == accelerator.gather(batch["label"])
            num_elems += accurate_preds.shape[0]
            accurate += accurate_preds.long().sum()

        eval_metric = accurate.item() / num_elems
        # Используем accelerator.print чтобы вывести только на главном процессе.
        accelerator.print(f"epoch {epoch}: {100 * eval_metric:.2f}")

Все, что осталось, — это использовать `notebook_launcher`.

Мы передаем функцию, аргументы (в виде кортежа) и количество процессов для обучения. (Дополнительную информацию см. в [документации](https://huggingface.co/docs/accelerate/package_reference/launchers#accelerate.notebook_launcher))

In [ ]:
import os
IS_COLAB_BACKEND = 'COLAB_GPU' in os.environ  # this is always set on Colab, the value is 0 or 1 depending on GPU presence
if IS_COLAB_BACKEND:
  args = ("fp16", 42, 64)
  print('найден GPU')
else:
  args = ("no", 42, 64)
  print('Не найден GPU')
try:
    device_name = os.environ['COLAB_TPU_ADDR']
    TPU_ADDRESS = 'grpc://' + device_name
    print('найден TPU на : {}'.format(TPU_ADDRESS))
except KeyError:
    print('TPU не найден')

найден GPU
TPU не найден


In [ ]:
from accelerate import notebook_launcher
from accelerate import Accelerator
from accelerate.utils import set_seed
from timm import create_model

In [ ]:
!pip install huggingface_hub

In [ ]:
args = ("no", 42, 64)
accelerator = None
accelerator = Accelerator(mixed_precision=args[0])
notebook_launcher(training_loop, args, num_processes=1)

Launching training on one GPU.
****************************************  accelerator.distributed_type DistributedType.NO


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/103M [00:00<?, ?B/s]

  0%|          | 0/1 [00:00<?, ?it/s]

epoch 0: 90.65


## Распределенный inference с Accelerate

Распределенный inference — распространенный вариант использования, когда вы хотите реализовать запуск модели на нескольких GPU. Пользователи часто хотят отправить несколько разных запросов, каждый на отдельный графический процессор, а затем получить результаты обратно.

### Проблема

Обычно при этом пользователи отправляют модель на определенное устройство, чтобы загрузить ее из CPU, а затем перемещают каждый промпт на другое устройство.

Базовый пайплайн, использующий библиотеку `diffusers`, может выглядеть примерно так:

In [ ]:
%%capture
!pip install diffusers==0.24.0 accelerate matplotlib timm tqdm  google-api-python-client>=1.12.5 -q
!pip install evaluate datasets==2.3.2 dill==0.3.5.1 evaluate==0.4.1 multiprocess==0.70.13 -q
!pip install torchvision responses==0.18.0 tokenizers==0.14.1 transformers==4.34.0 xxhash==3.4.1 -q
! pip install cloud-tpu-client==0.10 huggingface-hub==0.19.4 torch==2.1.2 diffusers -q

In [ ]:
!pip install --upgrade huggingface_hub

In [ ]:
import torch
import torch.distributed as dist
from diffusers import DiffusionPipeline

pipeline = DiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5",low_cpu_mem_usage=False)
pipeline("An image of a squirrel in Picasso style").images[0]

Затем следует выполнение вывода на основе конкретного промпта:

In [ ]:
def run_inference(rank, world_size):
    dist.init_process_group("nccl", rank=rank, world_size=world_size)
    pipe.to(rank)

    if torch.distributed.get_rank() == 0:
        prompt = "a dog"
    elif torch.distributed.get_rank() == 1:
        prompt = "a cat"

    result = pipe(prompt).images[0]
    result.save(f"result_{rank}.png")

Можно заметить, что нам приходится проверять ранг, чтобы знать, какой промпт отправить, что может быть немного утомительно.

Тогда пользователь также может подумать, что с `Accelerate` использование `Accelerator` для подготовки `dataloader` для такой задачи также может быть простым способом справиться с этим.

Сможет ли оно справиться с этим? Да. Однако добавляет ли это ненужный дополнительный код: тоже да.

### Решение


С помощью `Accelerate` мы можем упростить этот процесс, используя контекстный менеджер `Accelerator.split_between_processes()` (который также существует в `PartialState` и `AcceleratorState`). Эта функция автоматически разделит любые данные, которые вы ей передаете (будь то промпт, набор тензоров, словарь предыдущих данных и т. д.) по всем процессам (с возможностью дополнения), чтобы вы могли их сразу использовать.

Давайте перепишем приведенный выше пример, используя этот контекстный менеджер:

In [ ]:
from accelerate import PartialState  # Может быть Accelerator или AcceleratorState
from diffusers import DiffusionPipeline

pipe = DiffusionPipeline.from_pretrained("google/ddpm-celebahq-256")
distributed_state = PartialState()
pipe.to(distributed_state.device)

# Предполагаем
try:
  with distributed_state.split_between_processes(["a dog", "a cat"]) as prompt:
      result = pipe(prompt).images[0]
      result.save(f"result_{distributed_state.process_index}.png")
except:
  args = ("no", 42, 64)
  accelerator = None
  accelerator = Accelerator(cpu=True)
  notebook_launcher(training_loop, args, num_processes=1)

А затем, чтобы запустить код, мы можем использовать `Accelerate`:

Если вы создали файл конфигурации для использования с помощью `accelerate config`:

In [ ]:
!accelerate launch distributed_inference.py

Если у вас есть конкретный файл конфигурации, который вы хотите использовать:

In [ ]:
!accelerate launch --config_file my_config.json distributed_inference.py

Или, если вы не хотите создавать какие-либо файлы конфигурации и запускать их на двух GPU:

In [ ]:
!accelerate launch --num_processes 2 distributed_inference.py

Теперь мы довольно легко сократили шаблонный код, необходимый для разделения этих данных, на несколько строк кода.

Но что, если у нас будет странное распределение подсказок по графическим процессорам? Например, что, если у нас есть 3 промпта, но только 2 GPU?

Под управлением контекстного менеджера первый GPU получит первые два запроса, а второй GPU — третий, гарантируя, что все запросы будут разделены и не потребуется никаких дополнительных затрат.

Однако что, если мы затем захотим что-то сделать с результатами всех GPU? (Скажем, собрать их все и выполнить какую-то постобработку). Вы можете передать `apply_padding=True`, чтобы гарантировать, что списки промптов заполняются до одинаковой длины, а дополнительные данные берутся из последней выборки. Таким образом, все GPU будут получать одинаковое количество запросов, и вы сможете собрать результаты.

Например:

In [ ]:
exit(0)
!pip install diffusers transformers accelerate

In [ ]:
!pip install numpy

In [ ]:
from diffusers import DDPMScheduler, UNet2DModel
from PIL import Image
import torch
import numpy as np

scheduler = DDPMScheduler.from_pretrained("google/ddpm-cat-256")
model = UNet2DModel.from_pretrained("google/ddpm-cat-256")
scheduler.set_timesteps(50)

sample_size = model.config.sample_size
noise = torch.randn((1, 3, sample_size, sample_size))
input = noise

for t in scheduler.timesteps:
    with torch.no_grad():
        noisy_residual = model(input, t).sample
        prev_noisy_sample = scheduler.step(noisy_residual, t, input).prev_sample
        input = prev_noisy_sample

image = (input / 2 + 0.5).clamp(0, 1)
image = image.cpu().permute(0, 2, 3, 1).numpy()[0]
image = Image.fromarray((image * 255).round().astype("uint8"))
image

In [ ]:
from diffusers import DiffusionPipeline
import torch

pipeline = DiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", torch_dtype=torch.float32)
pipeline("An image of a squirrel in Picasso style").images[0]

In [ ]:
!pip install accelerate

In [ ]:
from diffusers import DiffusionPipeline
from accelerate import PartialState
from PIL import Image

pipe = DiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16)
distributed_state = PartialState()
pipe.to(distributed_state.device)

# Предполагаем два процесса
with distributed_state.split_between_processes(["a dog", "a cat", "a chicken"], apply_padding=True) as prompt:
    piperesult1=pipe(f"An image of a {prompt} in Picasso style,(3D)").images[0]
    piperesult2=pipe(f"An image of a {prompt} in Picasso style,((low res))").images[0]

piperesult1
piperesult2


На первом GPU промптами будут ["a dog", "a cat"], а на втором GPU - ["a chicken", "a chicken"]. Обязательно отбросьте окончательный образец, так как он будет копией предыдущего.

### Вывод

В этом блокноте показано, как выполнять распределенное обучение изнутри Jupyter Notebook. Некоторые ключевые замечания, которые следует запомнить:

* Обязательно сохраните любой код, использующий CUDA (или импорт CUDA) для функции, передаваемой в `notebook_launcher`.
* Установите `num_processes` как количество устройств, используемых для обучения (например, количество GPU, CPU, TPU и т.д.).

<hr>

Один и тот же сценарий может быть запущен в любой из следующих конфигураций:

- один процессор CPU или один графический
процессор GPU
- с несколькими графическими процессорами GPU (с использованием распределенного режима PyTorch)
- с несколькими процессорами TPU
- fp16 (смешанная точность) или fp32 (нормальная точность)


Чтобы запустить его в каждом из этих различных режимов, используйте следующие команды:
- один центральный процессор CPU:
    * с сервера без графического процессора GPU
        ```bash
        python ./nlp_example.py
        ```
    * с любого сервера, передав `cpu=True` в `Accelerator`.
        ```bash
        python ./nlp_example.py --cpu
        ```
    * с любого сервера, используя Accelerate
        ```bash
        accelerate launch --cpu ./nlp_example.py
        ```
- один GPU:
    ```bash
    python ./nlp_example.py  # from a server with a GPU
    ```
- fp16 (смешанная точность)
    * с любого сервера, передав `mixed_precison=fp16` в Accelerator.
        ```bash
        python ./nlp_example.py --mixed_precision fp16
        ```
    * с любого сервера с Accelerator
        ```bash
        accelerate launch --mixed_precision fp16 ./nlp_example.py
        ```
- с несколькими графическими процессорами GPU (с использованием распределенного режима PyTorch) на 1 машине
    * используя Accelerate config

        ```bash
        accelerate config accelerate launch ./nlp_example.py
        ```
        
    * с обычным PyTorch  (можно брать `torch.distributed.launch` со старыми версиями PyTorch)
        ```bash
        python -m torchrun --nproc_per_node 2 --use_env ./nlp_example.py
        ```
- с несколькими графическими процессорами GPU (с использованием распределенного режима PyTorch) на нескольких машинах
    * используя Accelerate config на каждой машине:
        ```bash
        accelerate config  # This will create a config file on each server
        accelerate launch ./nlp_example.py  # This will run the script on each server
        ```
    * только PyTorch ( можно брать`torch.distributed.launch` со старыми версиями PyTorch)
        ```bash
        python -m torchrun --nproc_per_node 2 \
            --use_env \
            --node_rank 0 \
            --master_addr master_node_ip_address \
            ./nlp_example.py  # On the first server
        python -m torchrun --nproc_per_node 2 \
            --use_env \
            --node_rank 1 \
            --master_addr master_node_ip_address \
            ./nlp_example.py  # On the second server
        ```
- один или несколько TPU
    * используя Accelerate config
        ```bash
        accelerate config  # This will create a config file on your TPU server
        accelerate launch ./nlp_example.py  # This will run the script on each server
        ```
    * используя PyTorch:
    
        Добавьте строку `xmp.spawn` в скрипт.

In [ ]:
#импорт 28 секунд
!pip install accelerate matplotlib timm tqdm
!pip install evaluate datasets==2.3.2 transformers

In [ ]:
!ls && pwd
!python /content/nlp_example.py --data_dir=sample_data --num_samples=10